<a href="https://colab.research.google.com/github/SmartEason/Esp32_yolo/blob/main/Gesture_Detection_Swift-YOLO_192.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <h1>Welcom to SSCMA for Google Colab Training Example 🔥 </h1>
  <a href="https://sensecraftma.seeed.cc/" target="_blank"><img width="20%" src="https://files.seeedstudio.com/sscma/docs/images/SSCMA-Hero.png"></a>
</div>

## ⚠️ Important Notice

Before you begin, please ensure:

1. **Enable GPU Acceleration**: Click `Runtime` → `Change runtime type` → Select `GPU` in `Hardware accelerator`
2. **Colab Version Requirement**: **You MUST use Colab version 2025.07** for compatibility. Other versions may cause unexpected errors.

If you encounter version-related issues, you can run the following commands in the first code cell to check your environment:

```python
# Check if GPU is available
!nvidia-smi

# Check Python version
!python --version
```

# Gesture Detection - Swift-YOLO

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seeed-studio/sscma-model-zoo/blob/main/notebooks/en/Gesture_Detection_Swift-YOLO_192.ipynb)

**Version:** 1.0.0

**Category:** Object Detection

**Algorithm:** [Swift-YOLO](configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py)

**Dataset:** [Gesture](https://universe.roboflow.com/rsp/paper-aaj0p/dataset/33)

**⚠️ IMPORTANT: To download the dataset, you need to set your Roboflow API key.**

1. Go to your [Roboflow Settings](https://app.roboflow.com/settings/api).
2. Copy your private API key.
3. In Colab, click the **key icon** on the **left sidebar** to open Secrets.
4. Add a new secret named `ROBOFLOW_API_KEY` with your API key as the value.

**Class:** `paper`, `rock`, `scissors`

![Gesture Detection](https://files.seeedstudio.com/sscma/static/detection_gesture.png)

The model is a Swift-YOLO model trained on the gesture detection dataset.



In [3]:
# 確保你在正確的目錄下
%cd /content/ModelAssistant

# 自動將檔案中的 torch==2.0.0 替換為相容 Python 3.12 的 torch>=2.2.0
!sed -i 's/torch==2.0.0/torch>=2.2.0/g' requirements/inference.txt

# 重新執行安裝指令
!pip install -r requirements/inference.txt

/content/ModelAssistant


In [1]:
# 建立一個全域修補程式，自動幫舊版套件補上缺失的 ImpImporter
patch_code = """
import pkgutil
import importlib.machinery
# 欺騙系統，將被廢棄的 ImpImporter 替換成現有的 FileFinder
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = importlib.machinery.FileFinder
"""

with open('/usr/local/lib/python3.12/dist-packages/sitecustomize.py', 'w') as f:
    f.write(patch_code)

print("修補程式植入完成！")

修補程式植入完成！


## ⚙️Prerequisites
### Setup SSCMA
Clone the [repository](https://github.com/Seeed-Studio/ModelAssistant) and install the dependencies.

In [2]:
!pip install ethos-u-vela
!pip install --upgrade setuptools
!git clone https://github.com/Seeed-Studio/ModelAssistant.git -b   #clone the repo
%cd ModelAssistant
!. ./scripts/setup_colab.sh



  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 60.2.0
    Uninstalling setuptools-60.2.0:
      Successfully uninstalled setuptools-60.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
openxlab 0.0.38 requires setuptools~=60.2.0, but you have setuptools 82.0.1 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


error: switch `b' requires a value
/content/ModelAssistant
Checking if CUDA available... OK
ERROR: Could not find a version that satisfies the requirement torch==2.0.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.0.0
Installing base deps... Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
^C


### Download the pretrain model weights file

In [ ]:
%mkdir -p Gesture_Detection_Swift-YOLO_192
!wget -c https://files.seeedstudio.com/sscma/model_zoo/detection/gesture/swift_yolo_1xb16_300e_coco_sha1_adda465db843aae8384c90c82e223c2cd931cad2.pth -O Gesture_Detection_Swift-YOLO_192/pretrain.pth

### Download the dataset

In [10]:
'''%mkdir -p Gesture_Detection_Swift-YOLO_192
!pip install roboflow
import os
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
from roboflow import download_dataset
dataset = download_dataset("https://universe.roboflow.com/rsp/paper-aaj0p/dataset/33", "coco")
dataset_path = dataset.location
import shutil
shutil.move(dataset_path, "Gesture_Detection_Swift-YOLO_192/dataset")
'''
# url = https://universe.roboflow.com/ds/EnbI6Gi5VK?key=y1VPYHBBu9
# %mkdir -p Gesture_Detection_Swift-YOLO_192/dataset
# !wget -c https://universe.roboflow.com/ds/EnbI6Gi5VK?key=y1VPYHBBu9 -O Gesture_Detection_Swift-YOLO_192/dataset.zip
# !unzip -q Gesture_Detection_Swift-YOLO_192/dataset.zip -d Gesture_Detection_Swift-YOLO_192/dataset
# roboflow provide code
!pip install roboflow
!pip install requests==2.32.4 tqdm>=4.67.0
from roboflow import Roboflow
rf = Roboflow(api_key="WXJe2c98Sw7ix3e2tI3K")
project = rf.workspace("rsp").project("paper-aaj0p")
version = project.version(36)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...


## 🚀Train a model with SSCMA
All the training parameters are in the `config.py` file, you can change the parameters to train your own model.

Below are explanations of some common parameters. You can also refer to the [documentation](https://sensecraftma.seeed.cc/tutorials/config) for more details.
- `data_root` - the datasets path.
- `epochs`- the train epochs. **we use 10 epochs as an example**.
- `batch_size` - the batch size.
- `height` - the image height.
- `width` - the image width.
- `load_from` - the pretrained model path.
- `num_classes` - the number of classes.

You can overwrite the parameters in the `config.py` file by using the `--cfg-options` argument.
```bash
# Example
sscma.train config.py --cfg-options data_root=./datasets/test_dataset epochs=10
```

In [ ]:
!sscma.train configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
--cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

## 📦Export the model
After training, you can export the model to the format for deployment. SSCMA supports exporting to ONNX, and TensorFlow Lite at present.
You can also refer to the [documentation](https://sensecraftma.seeed.cc/tutorials/export/overview) for more details.

```bash
python3 tools/export.py \
    "<CONFIG_FILE_PATH>" \
    "<CHECKPOINT_FILE_PATH>"
```

In [ ]:
import os
with open('Gesture_Detection_Swift-YOLO_192/last_checkpoint', 'r') as f:
	os.environ['CHECKPOINT_FILE_PATH'] = f.read()

In [ ]:
!sscma.export configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py $CHECKPOINT_FILE_PATH --cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

### 📝Evaluate the model
After exporting the model, you can evaluate the model on the test dataset.
You can also refer to the [documentation](https://sensecraftma.seeed.cc/tutorials/export/overview) for more details.


```bash
python3 tools/inference.py \
    "<CONFIG_FILE_PATH>" \
    "<CHECKPOINT_FILE_PATH>"
```

### Evaluate the PyTorch model

In [ ]:
!sscma.inference configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py ${CHECKPOINT_FILE_PATH%.*}.pth \
--cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

### Evaluate the ONNX model

In [ ]:
!sscma.inference configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py ${CHECKPOINT_FILE_PATH%.*}_float32.onnx \
--cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

### Evaluate the TFLite FLOAT32 model

In [ ]:
!sscma.inference configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py ${CHECKPOINT_FILE_PATH%.*}_float32.tflite \
--cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

### Evaluate the TFLite INT8 model

In [ ]:
!sscma.inference configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py ${CHECKPOINT_FILE_PATH%.*}_int8.tflite \
--cfg-options  \
    work_dir=Gesture_Detection_Swift-YOLO_192 \
    num_classes=3 \
    epochs=10  \
    height=192 \
    width=192 \
    data_root=Gesture_Detection_Swift-YOLO_192/dataset/ \
    load_from=Gesture_Detection_Swift-YOLO_192/pretrain.pth

## 🤖 Deploy the model
After model training, evaluation and export, you can deploy the model to your device. You can refer to [Documentation](https://sensecraftma.seeed.cc/deploy/overview) for more details.

In [ ]:
%ls -lh Gesture_Detection_Swift-YOLO_192/

### Thanks for Trying Out SSCMA 🎉

Congratulations, you have completed this tutorial. If you are interested in more application scenarios or our projects, please feel free to give [SSCMA](https://github.com/Seeed-Studio/ModelAssistant) a star ✨ on GitHub.

If you have any questions about this tutorial, please also feel free to [submit an issue](https://github.com/Seeed-Studio/ModelAssistant/issues).